<a href="https://colab.research.google.com/github/RDRamosU/mercado-laboral-steam-cr/blob/main/notebooks/02_exploracion_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto 2 — Mercado Laboral Tech en Costa Rica
## Notebook 02 — Exploración de datos

**Autor:** Ruben Dario Ramos Ulate
**Fecha:** Junio 2026  

---

## 1. Instalación e importación de librerías

In [37]:
!pip install pdfplumber openpyxl -q

import pandas as pd
import numpy as np
import pdfplumber
import requests
import os

print ("Librerías importadas ✓")
print (f"pandas: {pd.__version__}")
print (f"numpy: {np.__version__}")

Librerías importadas ✓
pandas: 2.2.2
numpy: 2.0.2


## 2. Carga del archivo ECE 1987–2025 (INEC)

Fuente: Encuesta Continua de Empleo — INEC  
Indicadores de empleo, desempleo y fuerza laboral 2014–2025

In [40]:
# Subir el archivo directamente desde tu computadora
from google.colab import files

print("Selecciona el archivo ECE desde tu computadora...")
uploaded = files.upload()

# Verificar qué se subió
for nombre, contenido in uploaded.items():
    print(f"Archivo subido: {nombre} — {len(contenido):,} bytes")
    nombre_ece = nombre

Selecciona el archivo ECE desde tu computadora...


Saving Principales+indicadores+EHPM+-+ECE+1987-2025+agrupado+Regiones+de+planficacin.xlsx to Principales+indicadores+EHPM+-+ECE+1987-2025+agrupado+Regiones+de+planficacin.xlsx
Archivo subido: Principales+indicadores+EHPM+-+ECE+1987-2025+agrupado+Regiones+de+planficacin.xlsx — 225,305 bytes


In [41]:
# Verificar que el archivo es un Excel válido
import zipfile

try:
    with zipfile.ZipFile(nombre_ece, 'r') as z:
        print(f"✓ Archivo válido — contiene {len(z.namelist())} elementos internos")
        print("Primeros elementos:", z.namelist()[:5])
except zipfile.BadZipFile:
    print("✗ El archivo está corrupto o no es un Excel válido")

✓ Archivo válido — contiene 47 elementos internos
Primeros elementos: ['[Content_Types].xml', '_rels/.rels', 'xl/_rels/workbook.xml.rels', 'xl/workbook.xml', 'xl/worksheets/sheet4.xml']


In [42]:
# Inspección de hojas disponibles
xl = pd.ExcelFile(nombre_ece, engine='openpyxl')
print("=== HOJAS DISPONIBLES ===")
for hoja in xl.sheet_names:
    print(f"  - {hoja}")

=== HOJAS DISPONIBLES ===
  - Nota técnica
  - Total
  - Hombres
  - Mujeres
  - Reg. Central
  - Reg. Chorotega
  - Reg. Pac. Central
  - Reg. Brunca
  - Reg. Hue. Atlántica
  - Reg. Hue. Norte


In [43]:
# Lectura de la hoja Total
df_ece_raw = pd.read_excel(
    nombre_ece,
    sheet_name="Total",
    header=None,
    engine='openpyxl'
)
print(f"Dimensiones: {df_ece_raw.shape}")
print("\nPrimeras 10 filas:")
print(df_ece_raw.iloc[:10, :8])

Dimensiones: (95, 40)

Primeras 10 filas:
                 0        1        2        3        4        5        6  \
0              NaN      NaN      NaN      NaN      NaN      NaN      NaN   
1            Total      NaN      NaN      NaN      NaN      NaN      NaN   
2      1987 - 2025      NaN      NaN      NaN      NaN      NaN      NaN   
3              NaN      NaN      NaN      NaN      NaN      NaN      NaN   
4              NaN     1987     1988     1989     1990     1991     1992   
5              NaN      NaN      NaN      NaN      NaN      NaN      NaN   
6  Población total  2799751  2876683  2960255  3029093  3099339  3168612   
7              NaN      NaN      NaN      NaN      NaN      NaN      NaN   
8             Sexo  2799751  2876683  2960255  3029093  3099339  3168612   
9          Hombres  1407004  1442947  1471144  1504669  1527373  1565347   

         7  
0      NaN  
1      NaN  
2      NaN  
3      NaN  
4     1993  
5      NaN  
6  3238595  
7      NaN  
8  3

In [44]:
# La fila 4 (índice 4) contiene los años
fila_años = 4
year_row = df_ece_raw.iloc[fila_años]

# Encontrar columnas de 2014 a 2025
año_a_col = {}
for col_idx, val in enumerate(year_row):
    if isinstance(val, (int, float)) and 2014 <= val <= 2025:
        año_a_col[int(val)] = col_idx

print("=== COLUMNAS POR AÑO (2014-2025) ===")
for año, col in año_a_col.items():
    print(f"  {año}: columna {col}")

=== COLUMNAS POR AÑO (2014-2025) ===
  2014: columna 28
  2015: columna 29
  2016: columna 30
  2017: columna 31
  2018: columna 32
  2019: columna 33
  2020: columna 34
  2021: columna 35
  2022: columna 36
  2023: columna 37
  2024: columna 38
  2025: columna 39


In [45]:
# Extraer todos los indicadores con sus valores 2014-2025
print("=== INDICADORES DISPONIBLES ===\n")

for i, row in df_ece_raw.iterrows():
    val = row.iloc[0]
    if pd.notna(val) and str(val).strip() and str(val).strip() != 'nan':
        vals = {año: row.iloc[col] for año, col in año_a_col.items()}
        # Solo mostrar filas con al menos un valor numérico
        tiene_datos = any(isinstance(v, (int, float)) for v in vals.values())
        if tiene_datos:
            print(f"Fila {i:2d}: {str(val).strip()}")
            muestra = {a: round(v, 1) if isinstance(v, float) else v
                      for a, v in list(vals.items())[-6:]}
            print(f"         Últimos años: {muestra}")

=== INDICADORES DISPONIBLES ===

Fila  1: Total
         Últimos años: {2020: nan, 2021: nan, 2022: nan, 2023: nan, 2024: nan, 2025: nan}
Fila  2: 1987 - 2025
         Últimos años: {2020: nan, 2021: nan, 2022: nan, 2023: nan, 2024: nan, 2025: nan}
Fila  6: Población total
         Últimos años: {2020: 5108539.2, 2021: 5161049.5, 2022: 5211340.5, 2023: 5260078.2, 2024: 5307827.5, 2025: 5353754.5}
Fila  8: Sexo
         Últimos años: {2020: 5108539.2, 2021: 5161049.5, 2022: 5211340.5, 2023: 5260078.2, 2024: 5307827.5, 2025: 5353754.5}
Fila  9: Hombres
         Últimos años: {2020: 2574194.8, 2021: 2599858.8, 2022: 2623958.8, 2023: 2649492.0, 2024: 2670325.0, 2025: 2692397.8}
Fila 10: Mujeres
         Últimos años: {2020: 2534344.5, 2021: 2561190.8, 2022: 2587381.8, 2023: 2610586.2, 2024: 2637502.5, 2025: 2661356.8}
Fila 12: Grupos edad
         Últimos años: {2020: 5108539.2, 2021: 5161049.5, 2022: 5211340.5, 2023: 5260078.2, 2024: 5307827.5, 2025: 5353754.5}
Fila 13: Población menor de

In [46]:
# Extraer indicadores clave directamente del archivo ECE
# Filas identificadas en la exploración anterior

filas_indicadores = {
    'poblacion_total':        6,
    'fuerza_trabajo':        17,
    'ocupados':              18,
    'desempleados':          19,
    'sector_primario':       27,
    'sector_secundario':     28,
    'sector_comercio_servicios': 29,
    'ocup_calificada_alta':  33,
    'ocup_calificada_media': 34,
    'ocup_no_calificada':    35,
    'sector_publico':        43,
    'sector_privado':        44,
    'educacion_universitaria': 61,
    'tasa_participacion':    68,
    'tasa_ocupacion':        69,
    'tasa_desempleo':        71,
    'pct_subempleo':         78,
}

# Construir DataFrame
data = {'año': list(año_a_col.keys())}

for nombre, fila in filas_indicadores.items():
    data[nombre] = [
        df_ece_raw.iloc[fila, col]
        for col in año_a_col.values()
    ]

df_ece = pd.DataFrame(data)

print("DataFrame ECE construido directamente del archivo ✓")
print(f"Filas: {len(df_ece)} | Columnas: {len(df_ece.columns)}")
print()
print(df_ece[['año', 'ocupados', 'desempleados',
              'tasa_desempleo', 'educacion_universitaria']].to_string(index=False))

DataFrame ECE construido directamente del archivo ✓
Filas: 12 | Columnas: 18

 año   ocupados  desempleados  tasa_desempleo  educacion_universitaria
2014 2064405.50     219736.00        9.617385                543533.25
2015 2057301.50     218802.00        9.612974                526981.25
2016 1995747.75     210431.25        9.537852                514245.25
2017 2051236.75     204610.25        9.071724                528944.00
2018 2117052.50     242591.25       10.264046                523217.00
2019 2175098.00     289858.00       11.755392                601180.25
2020 1938173.00     468359.50       19.606885                564117.75
2021 2039832.00     400938.50       16.433372                597963.00
2022 2154252.25     299770.75       12.218831                612925.50
2023 2094838.50     205058.00        8.890874                616583.25
2024 2200452.00     176986.00        7.452444                687214.50
2025 2188736.25     158246.00        6.739376                705863.00

In [47]:
# Verificación de consistencia
# ocupados + desempleados debe aproximarse a fuerza_trabajo
df_ece['verificacion_ft'] = (
    (df_ece['ocupados'] + df_ece['desempleados'] -
     df_ece['fuerza_trabajo']).abs() < 1000
)

print("=== VERIFICACIÓN DE CONSISTENCIA ===")
print("ocupados + desempleados ≈ fuerza_trabajo\n")
print(df_ece[['año', 'ocupados', 'desempleados',
              'fuerza_trabajo', 'verificacion_ft']].to_string(index=False))
print(f"\nTodos consistentes: {df_ece['verificacion_ft'].all()}")

=== VERIFICACIÓN DE CONSISTENCIA ===
ocupados + desempleados ≈ fuerza_trabajo

 año   ocupados  desempleados  fuerza_trabajo  verificacion_ft
2014 2064405.50     219736.00      2284141.50             True
2015 2057301.50     218802.00      2276103.50             True
2016 1995747.75     210431.25      2206179.00             True
2017 2051236.75     204610.25      2255847.00             True
2018 2117052.50     242591.25      2359643.75             True
2019 2175098.00     289858.00      2464956.00             True
2020 1938173.00     468359.50      2406532.50             True
2021 2039832.00     400938.50      2440770.50             True
2022 2154252.25     299770.75      2454023.00             True
2023 2094838.50     205058.00      2299896.50             True
2024 2200452.00     176986.00      2377438.00             True
2025 2188736.25     158246.00      2346982.25             True

Todos consistentes: True


In [48]:
# Calcular indicadores derivados útiles para el análisis
df_ece['pct_educacion_universitaria'] = (
    df_ece['educacion_universitaria'] /
    df_ece['ocupados'] * 100
).round(1)

df_ece['pct_sector_servicios'] = (
    df_ece['sector_comercio_servicios'] /
    df_ece['ocupados'] * 100
).round(1)

df_ece['pct_ocup_calificada_alta'] = (
    df_ece['ocup_calificada_alta'] /
    df_ece['ocupados'] * 100
).round(1)

print("=== INDICADORES DERIVADOS ===\n")
print(df_ece[['año', 'tasa_desempleo', 'pct_educacion_universitaria',
              'pct_sector_servicios', 'pct_ocup_calificada_alta']].to_string(index=False))

=== INDICADORES DERIVADOS ===

 año  tasa_desempleo  pct_educacion_universitaria  pct_sector_servicios  pct_ocup_calificada_alta
2014        9.617385                         26.3                  71.0                      21.1
2015        9.612974                         25.6                  68.4                      21.6
2016        9.537852                         25.8                  69.0                      22.1
2017        9.071724                         25.8                  68.9                      19.7
2018       10.264046                         24.7                  68.0                      19.4
2019       11.755392                         27.6                  69.2                      22.2
2020       19.606885                         29.1                  68.4                      21.9
2021       16.433372                         29.3                  69.6                      23.3
2022       12.218831                         28.5                  70.3                

In [49]:
# Exportar dataset ECE procesado
df_ece.to_csv("ece_indicadores_2014_2025.csv", index=False)
print("Dataset ECE exportado: ece_indicadores_2014_2025.csv ✓")
print(f"Registros: {len(df_ece)}")
print(f"Columnas: {list(df_ece.columns)}")

Dataset ECE exportado: ece_indicadores_2014_2025.csv ✓
Registros: 12
Columnas: ['año', 'poblacion_total', 'fuerza_trabajo', 'ocupados', 'desempleados', 'sector_primario', 'sector_secundario', 'sector_comercio_servicios', 'ocup_calificada_alta', 'ocup_calificada_media', 'ocup_no_calificada', 'sector_publico', 'sector_privado', 'educacion_universitaria', 'tasa_participacion', 'tasa_ocupacion', 'tasa_desempleo', 'pct_subempleo', 'verificacion_ft', 'pct_educacion_universitaria', 'pct_sector_servicios', 'pct_ocup_calificada_alta']


## 6. Hallazgos preliminares de la exploración

### ECE — Mercado laboral general 2014–2025

- **Recuperación post-pandemia notable:** la tasa de desempleo bajó de 19.6%
  en 2020 a 7.5% en 2024, la más baja del periodo analizado.
- **Crecimiento del talento universitario:** los ocupados con educación
  universitaria crecieron de forma sostenida, pasando de ~564.118 en 2020
  a 687.215 en 2024 (+21.8%).
- **Sector servicios dominante:** concentra consistentemente más del 65%
  del empleo total, el ecosistema natural donde opera el sector tech.
- **Ocupación calificada alta en alza:** creció de 425.170 en 2020 a
  532.901 en 2024, reflejando la sofisticación creciente del mercado.

### Radiografía Laboral CONARE — Desempleo STEM vs nacional

- Desempleo STEM 2022: **4.2%** vs desempleo nacional **12.2%**
- Computación 2019: desempleo entre **0% y 4.6%**
- La brecha confirma: el mercado absorbe casi todos los graduados tech

### Empleo tech — MICITT, BCCR, PROCOMER

- Empleo TIC formal: **+37.3%** entre 2020 y 2021
- Empleo ZF servicios: creció de 90.829 a 119.982 entre 2020 y 2024
- Salario ZF 2024: **USD 2.319/mes** vs USD 1.169 nacional → **2x más alto**

Estos hallazgos confirman la hipótesis del Notebook 01:
**la brecha es de escasez de oferta, no de exceso de graduados sin empleo.**

In [52]:
print("=" * 55)
print("RESUMEN — NOTEBOOK 02 COMPLETADO")
print("=" * 55)
print(f"\nDatasets construidos:")
print(f"  df_ece          → {df_ece.shape[0]} filas · {df_ece.shape[1]} columnas")
print(f"  df_radiografia  → 12 filas · 6 columnas (construido en celda 14)")
print(f"  df_empleo_tech  → 5 filas · 8 columnas (construido en celda 16)")
print(f"\nArchivos exportados:")
print(f"  ece_indicadores_2014_2025.csv ✓")
print(f"\nSiguiente paso: Notebook 03 — Limpieza y preparación")

RESUMEN — NOTEBOOK 02 COMPLETADO

Datasets construidos:
  df_ece          → 12 filas · 22 columnas
  df_radiografia  → 12 filas · 6 columnas (construido en celda 14)
  df_empleo_tech  → 5 filas · 8 columnas (construido en celda 16)

Archivos exportados:
  ece_indicadores_2014_2025.csv ✓

Siguiente paso: Notebook 03 — Limpieza y preparación
